# ByteRCNN — FFT-75 Scenario #1 Baseline

**Frozen benchmark**: 512-byte fragments · 75 classes · official pre-split NPZ files

**Dataset source**: Kaggle Dataset (mounted at `/kaggle/input/`) — no download or extraction needed.

Pipeline:
1. Install dependencies
2. Clone repo (`eval-byteRCNN` branch)
3. Configure paths (Kaggle Dataset slug + output dirs)
4. ~~Download FFT-75.zip~~ — **SKIPPED** (using Kaggle Dataset mount)
5. ~~Unzip archive~~ — **SKIPPED** (using Kaggle Dataset mount)
6. Verify dataset integrity
7. Sanity training (2 epochs, 2 000 samples)
8. Full training (30 epochs, early stopping, AMP)
9. Evaluation on frozen test set
10. Zip and save outputs

> **Only one variable to change before running:**
> ```python
> KAGGLE_DATASET_SLUG = 'your-username/your-dataset-name'
> ```

## Cell 1 — Install Dependencies

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

pip('pyyaml>=6.0')
pip('scikit-learn>=1.5')
pip('pandas>=2.2')
pip('numpy>=1.26')
pip('matplotlib>=3.9')
pip('seaborn>=0.13')
pip('tqdm>=4.66')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## Cell 2 — Clone Repo

In [ ]:
import os, sys, shutil
from pathlib import Path

WORKING      = Path('/kaggle/working')
REPO_DIR     = WORKING / 'deepcarv'
BRANCH_NAME  = 'eval-byteRCNN'

# For private repos: set GITHUB_TOKEN to a Personal Access Token with read access.
# For public repos: leave as empty string and the plain HTTPS URL will be used.
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxx'

# Force fresh clone if src/ is missing (handles stale / wrong-branch clones)
if not REPO_DIR.exists() or not (REPO_DIR / 'deepcarv' / 'src').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    if GITHUB_TOKEN:
        clone_url = f'https://{GITHUB_TOKEN}@github.com/yuvnahr/deepcarv.git'
    else:
        clone_url = 'https://github.com/yuvnahr/deepcarv.git'
    !git clone --depth 1 --branch {BRANCH_NAME} {clone_url} {REPO_DIR}
else:
    print('Repo already present:', REPO_DIR)

# Add the inner deepcarv/ package root (where src/ lives) to sys.path
pkg_root = str(REPO_DIR / 'deepcarv')
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)

# Tell paths.py we are in Kaggle runtime
os.environ['KAGGLE_RUNTIME'] = '1'

print('sys.path[0]:', sys.path[0])
print('src exists :', (REPO_DIR / 'deepcarv' / 'src').exists())

## Cell 3 — Configure Kaggle Dataset

In [ ]:
KAGGLE_DATASET_SLUG = 'your-username/your-dataset-name'  # e.g. 'yuvnahr/fft-75-512'
# ─────────────────────────────────────────────────────────────────────────────

FRAGMENT_SIZE = 512   # FROZEN — do not change for Scenario #1

# Dataset is mounted read-only at /kaggle/input/<dataset-name>/
FFT75_DIR   = Path('/kaggle/input') / KAGGLE_DATASET_SLUG.split('/')[-1] / 'FFT-75'

# Writeable output directories
CKPT_DIR    = WORKING / 'checkpoints'
OUTPUTS_DIR = WORKING / 'outputs'
LOGS_DIR    = WORKING / 'logs'

for d in [CKPT_DIR, OUTPUTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths configured:')
for name, p in [('FFT75_DIR', FFT75_DIR), ('CKPT_DIR', CKPT_DIR),
                ('OUTPUTS_DIR', OUTPUTS_DIR), ('LOGS_DIR', LOGS_DIR)]:
    print(f'  {name:<15} {p}')

# Verify dataset structure
print('\nDataset structure check:')
all_ok = True
for split in ['train', 'val', 'test']:
    npz = FFT75_DIR / str(FRAGMENT_SIZE) / f'{split}.npz'
    status = '✅' if npz.exists() else '❌ MISSING'
    print(f'  FFT-75/{FRAGMENT_SIZE}/{split}.npz  {status}')
    if not npz.exists():
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        f'Dataset not found at {FFT75_DIR}\n'
        'Make sure you added your Kaggle dataset via "+ Add Data" and '
        'set KAGGLE_DATASET_SLUG correctly.'
    )

## Cell 4 — Verify Dataset

Checks NPZ keys, shapes, fragment lengths, and class counts.
**Stops immediately if anything is wrong.**

In [ ]:
from src.data.verify_dataset import verify_dataset

verify_dataset(
    data_dir=FFT75_DIR,
    fragment_size=FRAGMENT_SIZE,
)
print('Dataset verification PASSED.')

## Cell 5 — Sanity Training (2 epochs · 2 000 samples)

In [ ]:
from src.training.sanity_train_bytercnn import main as sanity_main

sanity_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--checkpoint_path', str(CKPT_DIR / 'sanity_bytercnn_fft75.pt'),
    '--epochs',    '2',
    '--subset',    '2000',
    '--batch_size','64',
    '--seed',      '42',
])

## Cell 6 — Full Training (30 epochs · early stopping · AMP)

In [ ]:
from src.training.train_bytercnn import main as train_main

# To resume from a checkpoint after an interruption, add:
#   '--resume', str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
# This will skip already-completed epochs and continue from where training stopped.

train_main([
    '--data_dir',        str(FFT75_DIR),
    '--fragment_size',   str(FRAGMENT_SIZE),
    '--checkpoint_path', str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--epochs',       '30',
    '--batch_size',   '512',
    '--lr',           '1e-3',
    '--patience',     '5',
    '--grad_clip',    '1.0',
    '--seed',         '42',
])

## Cell 7 — Evaluation on Frozen Test Set

In [ ]:
from src.evaluation.evaluate_bytercnn import main as eval_main

EVAL_OUT = OUTPUTS_DIR / 'bytercnn_fft75'

eval_main([
    '--checkpoint',    str(CKPT_DIR / 'best_bytercnn_fft75.pt'),
    '--data_dir',      str(FFT75_DIR),
    '--fragment_size', str(FRAGMENT_SIZE),
    '--out_dir',       str(EVAL_OUT),
    '--batch_size',    '256',
    '--seed',          '42',
])

# Quick display
import json
with open(EVAL_OUT / 'metrics.json') as f:
    m = json.load(f)
print('\n=== Key Metrics ===')
for k in ['accuracy','macro_f1','weighted_f1','time_per_sample_ms','peak_gpu_memory_mb']:
    print(f'  {k:<35} {m[k]:.4f}')

## Cell 8 — Zip and Save Outputs

In [ ]:
import shutil

bundle_dir  = WORKING / 'ByteRCNN_baseline_bundle'
bundle_dir.mkdir(exist_ok=True)

# Copy evaluation outputs
for fname in ['metrics.json','confusion_matrix.csv','per_class_metrics.csv',
              'predictions.csv','eval_summary.txt']:
    src = EVAL_OUT / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / fname)

# Copy training curve
curve = OUTPUTS_DIR / 'bytercnn_fft75' / 'training_curves.png'
if curve.exists():
    shutil.copy2(curve, bundle_dir / 'training_curves.png')

# Copy best checkpoint
best_ckpt = CKPT_DIR / 'best_bytercnn_fft75.pt'
if best_ckpt.exists():
    shutil.copy2(best_ckpt, bundle_dir / 'best_bytercnn_fft75.pt')

# Zip
zip_path = WORKING / 'ByteRCNN_baseline.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)

print(f'Output bundle → {zip_path}')
print(f'Size          : {zip_path.stat().st_size / 1e6:.1f} MB')
print('\nFiles in bundle:')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')